In [7]:
from huggingface_hub import login
from dotenv import load_dotenv
import os

load_dotenv()

login(token=os.getenv("HF_TOKEN"))

Note: Environment variable`HF_TOKEN` is set and is the current active token independently from the token you've just configured.


In [8]:
from datasets import load_dataset, Dataset, DatasetDict

ds = load_dataset("austindavis/lichess-uci", "201505")["train"]

# First cut: split off test set — this is quarantined until the end
train_val = ds.train_test_split(test_size=0.05, seed=42)

# Second cut: split train into train and validation
train_val_split = train_val["train"].train_test_split(test_size=0.05, seed=42)

final_splits = {
    "train": train_val_split["train"],
    "val":   train_val_split["test"],
    "test":  train_val["test"],
}

ds = DatasetDict(final_splits)
ds

DatasetDict({
    train: Dataset({
        features: ['Event', 'Site', 'White', 'Black', 'Result', 'UTCDate', 'UTCTime', 'WhiteElo', 'BlackElo', 'WhiteRatingDiff', 'BlackRatingDiff', 'ECO', 'Opening', 'TimeControl', 'Termination', 'Transcript'],
        num_rows: 1929144
    })
    val: Dataset({
        features: ['Event', 'Site', 'White', 'Black', 'Result', 'UTCDate', 'UTCTime', 'WhiteElo', 'BlackElo', 'WhiteRatingDiff', 'BlackRatingDiff', 'ECO', 'Opening', 'TimeControl', 'Termination', 'Transcript'],
        num_rows: 101534
    })
    test: Dataset({
        features: ['Event', 'Site', 'White', 'Black', 'Result', 'UTCDate', 'UTCTime', 'WhiteElo', 'BlackElo', 'WhiteRatingDiff', 'BlackRatingDiff', 'ECO', 'Opening', 'TimeControl', 'Termination', 'Transcript'],
        num_rows: 106878
    })
})

In [9]:
RESULT_STRINGS = {"1-0", "0-1", "1/2-1/2", "*"}

def clean_transcript(sample):
    cleaned_transcripts = []
    cleaned_moves = []
    num_moves = []
    
    for transcript in sample["Transcript"]:
        # Remove result string from end if present
        moves = transcript.lower().split()
        if moves and moves[-1] in RESULT_STRINGS:
            moves = moves[:-1]
        cleaned_transcripts.append(" ".join(moves))
        cleaned_moves.append(moves)
        num_moves.append(len(moves))
    
    return {
        "Transcript": cleaned_transcripts,
        "Moves": cleaned_moves,
        "Number of Moves": num_moves
    }

ds = ds.map(clean_transcript, batched=True, num_proc=2, desc="Cleaning transcripts")
ds["train"][0]

{'Event': 'Rated Bullet tournament https://lichess.org/tournament/MJHHFtgM',
 'Site': 'PPEvEQ91',
 'White': 'Suninheart',
 'Black': 'aila',
 'Result': '0-1',
 'UTCDate': datetime.date(2015, 5, 14),
 'UTCTime': datetime.time(21, 35, 10),
 'WhiteElo': 0,
 'BlackElo': 0,
 'WhiteRatingDiff': -9.0,
 'BlackRatingDiff': 4.0,
 'ECO': 'D05',
 'Opening': "Queen's Pawn Game, Zukertort Variation",
 'TimeControl': '120+0',
 'Termination': 'Normal',
 'Transcript': 'd2d4 d7d5 e2e3 e7e6 g1f3 g8f6 b1d2 c7c5 b2b3 c5d4 e3d4 b8c6 c1b2 f8e7 a2a3 e8g8 f1d3 a7a6 e1g1 c8d7 f3e5 a8b8 d2f3 c6e5 d4e5 f6e8 c2c4 d5c4 b3c4 f7f6 e5f6 e7f6 d1c2 h7h6 d3h7 g8h8 b2f6 d8f6 h7e4 e8d6 a1d1 d6e4 c2e4 d7c6 e4g4 c6f3 g2f3 f6f3 g4f3 f8f3 g1g2 f3a3 d1d2 a3c3 f1d1 c3c4 d2d8 b8d8 d1d8 h8h7 d8b8 b7b5 b8a8 c4a4 g2f3 b5b4 a8b8 a6a5 f3e3 a4a2 f2f4 a2h2 e3e4 h2h3 e4e5 b4b3 e5e6 a5a4 e6f5 a4a3 f5g4 h3c3 g4f5 a3a2 b8a8 b3b2 a8a2 b2b1q f5e5 b1a2 f4f5 a2e2 e5f4',
 'Moves': ['d2d4',
  'd7d5',
  'e2e3',
  'e7e6',
  'g1f3',
  'g8f6',
  'b1d2

We evaluate certain statistics about the games played, specifically regarding their number of moves.

Mean and variance

Code from: https://en.wikipedia.org/wiki/Algorithms_for_calculating_variance#Welford's_online_algorithm

In [10]:
import numpy as np
lengths = np.array(ds["train"]["Number of Moves"])
print(f"Mean: {lengths.mean():.2f}")
print(f"Median: {np.median(lengths):.2f}")
print(f"SD (sample): {lengths.std(ddof=1):.2f}")
print(f"5th percentile: {np.percentile(lengths, 5):.2f}")
print(f"25th percentile: {np.percentile(lengths, 25):.2f}")
print(f"75th percentile: {np.percentile(lengths, 75):.2f}")
print(f"95th percentile: {np.percentile(lengths, 95):.2f}")
print(f"Max: {lengths.max()}")
print(f"Min: {lengths.min()}")

Mean: 65.74
Median: 63.00
SD (sample): 31.43
5th percentile: 18.00
25th percentile: 45.00
75th percentile: 84.00
95th percentile: 122.00
Max: 600
Min: 0


In [ ]:
import chess
import chess.pgn